In [ ]:
import shutil
import os

# Define the specific file and folders
source_file = r'/kaggle/input/acc-sample-50-01/config.py'
destination_folder = r'/kaggle/working/'

# Create destination folder if it doesn't exist
os.makedirs(destination_folder, exist_ok=True)

# Define destination path
destination_path = os.path.join(destination_folder, os.path.basename(source_file))

# Copy the file
shutil.copy2(source_file, destination_path)

print(f"Copied {source_file} to {destination_path}")

In [ ]:
import os

config_file_path = 'config.py'
new_image_size = (224, 224)
new_clip_length = 32
new_batch_size = 8# A safe starting point for ViT
new_lr = 0.0001
new_patience = 15

print("--- Modifying config.py for the Vision Transformer (ViT) experiment ---")

if not os.path.exists(config_file_path):
    print(f"FATAL: The file '{config_file_path}' was not found.")
else:
    with open(config_file_path, 'r') as f:
        lines = f.readlines()

    updates = {'IMAGE_SIZE': False, 'CLIP_LENGTH': False, 'BATCH_SIZE': False}

    for i, line in enumerate(lines):
        if line.strip().startswith('IMAGE_SIZE'):
            lines[i] = f'IMAGE_SIZE = {new_image_size}   # H, W for the model\n'
            updates['IMAGE_SIZE'] = True
        elif line.strip().startswith('CLIP_LENGTH'):
            lines[i] = f'CLIP_LENGTH = {new_clip_length}          # Frames per clip\n'
            updates['CLIP_LENGTH'] = True
        elif line.strip().startswith('BATCH_SIZE'):
            lines[i] = f'BATCH_SIZE = {new_batch_size}\n'
            updates['BATCH_SIZE'] = True
        elif line.strip().startswith('LEARNING_RATE'):
            lines[i] = f'LEARNING_RATE = {new_lr}\n'
            updates['LEARNING_RATE'] = True
        elif line.strip().startswith('PATIENCE'):
            lines[i] = f'PATIENCE = {new_patience}\n'
            updates['PATIENCE'] = True

    if all(updates.values()):
        with open(config_file_path, 'w') as f:
            f.writelines(lines)
        print("✓ Successfully updated 'config.py'.")
    else:
        print("Warning: Could not find all required settings.")

print("\n--- Verifying final content of config.py ---")
with open(config_file_path, 'r') as f:
    print(f.read())

In [4]:
import torch

# Clear GPU cache
torch.cuda.empty_cache()

In [5]:
!pip install -q timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 67.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 4.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 71.2 MB/s eta 0:00:00:00:0100:01


In [18]:
!pip install ultralytics supervision

In [88]:
# step_4_prepare_multimodal_dataset.py (Version with Rich Features)
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from itertools import combinations
import config

# --- Configuration for this Step ---
config.VIDEO_INPUT_DIR = "/kaggle/input/acc-sample-50/"
config.TRACKING_OUTPUT_DIR = "/kaggle/input/tracking-data-output/"
MASTER_LABELS_PATH = "/kaggle/input/master-ground-truth-events/master_ground_truth_events.csv"
PREPARED_DATA_DIR = f"{config.OUTPUT_DIR}/prepared_dataset/"

NUM_EASY_NEGATIVE_SAMPLES = 4
NUM_HARD_NEGATIVE_SAMPLES = 5

def extract_clip_frames(video_path, start_frame, clip_length, output_dir, image_size):
    output_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): return False
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if start_frame < 0 or start_frame + clip_length > total_frames:
        cap.release(); return False
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    for i in range(clip_length):
        ret, frame = cap.read()
        if not ret: break
        frame_resized = cv2.resize(frame, image_size)
        cv2.imwrite(str(output_dir / f"{i:03d}.jpg"), frame_resized)
    cap.release()
    return True

def main():
    print("--- Step 4: Preparing Dataset with RICH Features ---")
    if not Path(MASTER_LABELS_PATH).exists():
        print(f"FATAL: Master labels file not found at {MASTER_LABELS_PATH}"); return

    labels_df = pd.read_csv(MASTER_LABELS_PATH)
    all_clip_manifest_data = []
    prepared_clips_dir = Path(PREPARED_DATA_DIR) / "clips"
    if prepared_clips_dir.exists():
        import shutil
        shutil.rmtree(prepared_clips_dir)
    prepared_clips_dir.mkdir(parents=True, exist_ok=True)

    for _, row in tqdm(labels_df.iterrows(), total=len(labels_df), desc="Processing Labeled Videos"):
        clip_name = row['clip_name']
        event_start, event_end = row['event_start_frame'], row['event_end_frame']
        video_path = Path(config.VIDEO_INPUT_DIR) / f"{clip_name}.mp4"
        track_path = Path(config.TRACKING_OUTPUT_DIR) / f"{clip_name}_tracks.csv"
        if not video_path.exists() or not track_path.exists(): continue

        track_df = pd.read_csv(track_path)
        if track_df.empty: continue

        # --- START: RICH FEATURE CALCULATION ---
        track_df = track_df.sort_values(by=['track_id', 'frame'])
        track_df['x_center'] = (track_df['x_min'] + track_df['x_max']) / 2
        track_df['y_center'] = (track_df['y_min'] + track_df['y_max']) / 2
        track_df[['dx', 'dy']] = track_df.groupby('track_id')[['x_center', 'y_center']].diff().fillna(0)
        track_df['velocity'] = np.sqrt(track_df['dx']**2 + track_df['dy']**2)
        track_df['deceleration'] = -track_df.groupby('track_id')['velocity'].diff().fillna(0)
        
        v_prev = track_df.groupby('track_id')[['dx', 'dy']].shift(1).fillna(0).to_numpy()
        v_curr = track_df[['dx', 'dy']].to_numpy()
        dot_product = np.sum(v_prev * v_curr, axis=1)
        norm_v_prev = np.linalg.norm(v_prev, axis=1)
        norm_v_curr = np.linalg.norm(v_curr, axis=1)
        denominator = norm_v_prev * norm_v_curr
        cosine_angle = np.divide(dot_product, denominator, out=np.ones_like(dot_product), where=denominator!=0)
        track_df['direction_change'] = np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))

        iou_data = []
        for frame_num, frame_df in track_df.groupby('frame'):
            max_iou = 0
            if len(frame_df) > 1:
                for i, j in combinations(frame_df.index, 2):
                    box1 = frame_df.loc[i, ['x_min', 'y_min', 'x_max', 'y_max']].values
                    box2 = frame_df.loc[j, ['x_min', 'y_min', 'x_max', 'y_max']].values
                    inter_x1, inter_y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
                    inter_x2, inter_y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
                    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
                    if inter_area > 0:
                        box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1]); box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
                        union_area = box1_area + box2_area - inter_area
                        max_iou = max(max_iou, inter_area / union_area if union_area > 0 else 0)
            iou_data.append({'frame': frame_num, 'max_iou': max_iou})
        
        track_df = pd.merge(track_df, pd.DataFrame(iou_data), on='frame', how='left')

        frame_features = track_df.groupby('frame').agg(
            max_deceleration=('deceleration', 'max'),
            max_iou=('max_iou', 'max'),
            max_direction_change=('direction_change', 'max')
        ).reset_index().set_index('frame')
        # --- END: RICH FEATURE CALCULATION ---

        def slice_and_save(start_f, label, clip_idx):
            clip_type = "crash" if label == 1 else "no_crash"
            clip_output_dir = prepared_clips_dir / clip_type / f"{clip_name}_{clip_idx}"
            if not extract_clip_frames(video_path, start_f, config.CLIP_LENGTH, clip_output_dir, config.IMAGE_SIZE): return
            end_f = start_f + config.CLIP_LENGTH
            clip_feature_df = frame_features.loc[start_f:end_f-1].copy()
            if len(clip_feature_df) < config.CLIP_LENGTH:
                 padding = pd.DataFrame(0, index=range(config.CLIP_LENGTH - len(clip_feature_df)), columns=clip_feature_df.columns)
                 clip_feature_df = pd.concat([clip_feature_df, padding])
            feature_path = clip_output_dir / "features.csv"
            clip_feature_df.to_csv(feature_path, index=False)
            all_clip_manifest_data.append({"clip_path": str(clip_output_dir), "feature_path": str(feature_path), "label": label})

        step = config.CLIP_LENGTH // 4
        for start_frame in range(event_start - step, event_end - config.CLIP_LENGTH + step, step):
            slice_and_save(start_frame, 1, f"crash_{start_frame}")
        
        safety_margin = config.CLIP_LENGTH * 2 
        max_easy_neg_start = event_start - safety_margin
        if max_easy_neg_start > 0:
            easy_neg_frames = np.linspace(0, max_easy_neg_start, NUM_EASY_NEGATIVE_SAMPLES, dtype=int)
            for i, start_frame in enumerate(easy_neg_frames): slice_and_save(start_frame, 0, f"easy_neg_{i}")

        for i in range(NUM_HARD_NEGATIVE_SAMPLES):
            start_frame = event_start - config.CLIP_LENGTH - (i * step)
            if start_frame >= 0: slice_and_save(start_frame, 0, f"hard_neg_{i}")
    
    manifest_df = pd.DataFrame(all_clip_manifest_data)
    manifest_path = Path(PREPARED_DATA_DIR) / "training_manifest.csv"
    manifest_df.to_csv(manifest_path, index=False)
    print(f"\n--- Data Preparation Complete ---"); print(f"Generated {len(manifest_df)} total clips.")
    print(f"  - Positive (crash) clips: {manifest_df['label'].sum()}"); print(f"  - Negative (no_crash) clips: {len(manifest_df) - manifest_df['label'].sum()}")
    print(f"Final training manifest saved to: {manifest_path}")

if __name__ == '__main__':
    main()

--- Step 4: Preparing Dataset with RICH Features ---


Processing Labeled Videos: 100%|██████████| 50/50 [03:16<00:00,  3.94s/it]


--- Data Preparation Complete ---
Generated 322 total clips.
  - Positive (crash) clips: 214
  - Negative (no_crash) clips: 108
Final training manifest saved to: /kaggle/working/output/prepared_dataset/training_manifest.csv


In [89]:
# step_5_train_vit_model.py (Version with Rich Feature Input)
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import config

# --- Setup ---
PREPARED_DATA_DIR = f"{config.OUTPUT_DIR}/prepared_dataset/"
MANIFEST_PATH = os.path.join(PREPARED_DATA_DIR, "training_manifest.csv")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using device: {device}")

# --- Dataset Class ---
class MultimodalDataset(Dataset):
    def __init__(self, manifest_df, transform=None):
        self.manifest_df = manifest_df
        self.transform = transform if transform else transforms.ToTensor()

    def __len__(self): return len(self.manifest_df)

    def __getitem__(self, idx):
        row = self.manifest_df.iloc[idx]
        frame_files = sorted(glob.glob(os.path.join(row['clip_path'], "*.jpg")))
        frames = [Image.open(f).convert("RGB") for f in frame_files]
        video_tensor = torch.stack([self.transform(frame) for frame in frames])
        
        feature_df = pd.read_csv(row['feature_path'])
        # --- KEY CHANGE: Load all three feature columns ---
        features = feature_df[['max_deceleration', 'max_iou', 'max_direction_change']].values.astype(np.float32)
        
        feature_tensor = torch.from_numpy(features)
        label = torch.tensor(row['label'], dtype=torch.float32)
        return (video_tensor, feature_tensor), label

# --- Model Definition ---
class VisionTransformerAccidentDetector(nn.Module):
    def __init__(self, num_features, hidden_size=256, dropout=0.5):
        super(VisionTransformerAccidentDetector, self).__init__()
        self.base_model = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=0, global_pool='avg')
        num_video_features = self.base_model.num_features
        for param in self.base_model.parameters(): param.requires_grad = False
        self.video_gru = nn.GRU(input_size=num_video_features, hidden_size=hidden_size, batch_first=True)
        self.feature_gru = nn.GRU(input_size=num_features, hidden_size=hidden_size // 4, batch_first=True)
        self.classifier = nn.Sequential(nn.Linear(hidden_size + hidden_size // 4, 512), nn.ReLU(), nn.Dropout(dropout), nn.Linear(512, 1))

    def forward(self, video_input, feature_input):
        batch_size, clip_len, C, H, W = video_input.shape
        video_input_reshaped = video_input.view(batch_size * clip_len, C, H, W)
        video_features = self.base_model(video_input_reshaped)
        video_features_seq = video_features.view(batch_size, clip_len, -1)
        _, video_hidden = self.video_gru(video_features_seq)
        _, feature_hidden = self.feature_gru(feature_input)
        video_hidden = video_hidden.squeeze(0); feature_hidden = feature_hidden.squeeze(0)
        fused = torch.cat((video_hidden, feature_hidden), dim=1)
        return self.classifier(fused).squeeze(1)

# --- Main Training Workflow ---
def main():
    print("--- Step 5: Training on Rich Feature Dataset ---")
    manifest_df = pd.read_csv(MANIFEST_PATH)
    train_df, val_df = train_test_split(manifest_df, test_size=0.2, random_state=42, stratify=manifest_df['label'])
    
    transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.3, contrast=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_dataset = MultimodalDataset(train_df, transform)
    val_dataset = MultimodalDataset(val_df, transform)
    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=2)
    
    class_weights = compute_class_weight('balanced', classes=np.unique(train_df['label']), y=train_df['label'])
    pos_weight = torch.tensor(class_weights[1] / class_weights[0], dtype=torch.float32).to(device)
    print(f"Using positive class weight: {pos_weight.item():.2f}")
    
    # --- KEY CHANGE: Tell the model to expect 3 input features ---
    model = VisionTransformerAccidentDetector(num_features=3).to(device)
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
    patience_counter, best_val_loss = 0, float('inf')
    
    for epoch in range(config.EPOCHS):
        model.train()
        train_loss = 0.0
        for (video, features), labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.EPOCHS} [Train]"):
            video, features, labels = video.to(device), features.to(device), labels.to(device)
            optimizer.zero_grad(); outputs = model(video, features); loss = criterion(outputs, labels)
            loss.backward(); optimizer.step(); train_loss += loss.item()

        model.eval()
        val_loss, val_corrects = 0.0, 0
        with torch.no_grad():
            for (video, features), labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{config.EPOCHS} [Val]"):
                video, features, labels = video.to(device), features.to(device), labels.to(device)
                outputs = model(video, features); loss = criterion(outputs, labels); val_loss += loss.item()
                preds = torch.sigmoid(outputs) > 0.5; val_corrects += torch.sum(preds == labels.byte())

        avg_train_loss = train_loss/len(train_loader); avg_val_loss = val_loss/len(val_loader); val_accuracy = val_corrects.double()/len(val_dataset)
        print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.4f}")
        
        if avg_val_loss < best_val_loss:
            print(f"Validation loss improved. Saving model..."); torch.save(model.state_dict(), config.SAVED_MODEL_PATH)
            best_val_loss = avg_val_loss; patience_counter = 0
        else:
            patience_counter += 1; print(f"Validation loss did not improve. Patience: {patience_counter}/{config.PATIENCE}")

        if patience_counter >= config.PATIENCE: print("Early stopping triggered."); break

    print(f"\n--- Training Complete ---"); print(f"Best model saved to: {config.SAVED_MODEL_PATH}")

if __name__ == "__main__":
    main()

✓ Using device: cuda
--- Step 5: Training on Rich Feature Dataset ---
Using positive class weight: 0.50


Epoch 1/50 [Val]: 100%|██████████| 17/17 [00:04<00:00,  3.61it/s]


Epoch 1: Train Loss: 0.4725, Val Loss: 0.4581, Val Acc: 0.6000
Validation loss improved. Saving model...


Epoch 2/50 [Val]: 100%|██████████| 17/17 [00:04<00:00,  3.51it/s]


Epoch 2: Train Loss: 0.4375, Val Loss: 0.4766, Val Acc: 0.7077
Validation loss did not improve. Patience: 1/10


...

Epoch 24: Train Loss: 0.1819, Val Loss: 0.3600, Val Acc: 0.8154
Validation loss did not improve. Patience: 8/10


Epoch 25/50 [Val]: 100%|██████████| 17/17 [00:04<00:00,  3.75it/s]


Epoch 25: Train Loss: 0.1366, Val Loss: 0.5649, Val Acc: 0.8462
Validation loss did not improve. Patience: 9/10


Epoch 26/50 [Val]: 100%|██████████| 17/17 [00:04<00:00,  3.62it/s]

Epoch 26: Train Loss: 0.1923, Val Loss: 0.4535, Val Acc: 0.8923
Validation loss did not improve. Patience: 10/10
Early stopping triggered.

--- Training Complete ---
Best model saved to: /kaggle/working/output/3d_cnn_accident_detector.pth


In [107]:
# ===================================================================================
# FINAL SCRIPT - STEP 6 (Definitive, Integrated Version)
# - Calculates real-time rich features (deceleration, IoU, direction change).
# - Adds Track ID labels to all bounding boxes.
# - Intelligently identifies and differentially colors involved vehicles.
# - Provides a live debugging printout of the model's confidence score.
# ===================================================================================

import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from collections import deque
from PIL import Image
from tqdm import tqdm
from itertools import combinations

import torch
import torch.nn as nn
from torchvision import transforms
import timm
from ultralytics import YOLO
import supervision as sv

# --- Configuration Class (All settings in one place) ---
class Config:
    # --- IMPORTANT: UPDATE THESE TWO PATHS ---
    # 1. Path to the .pth model file you trained with rich features.
    SAVED_MODEL_PATH = "/kaggle/working/output/3d_cnn_accident_detector.pth"
    # 2. Path to the new video you want to analyze.
    INPUT_VIDEO_PATH = "/kaggle/input/test-acc/vid-(192).mp4"
    
    # --- Standard Parameters ---
    YOLO_MODEL_PATH = 'yolov5x.pt'
    OUTPUT_VIDEO_PATH = "/kaggle/working/final_analysis_output.mp4"
    CLIP_LENGTH = 32
    IMAGE_SIZE = (224, 224)
    
    # --- Analysis Parameters (Tune these based on the debug output) ---
    DETECTION_THRESHOLD = 0.65  # A good starting point for a well-trained model.
    PREDICTION_INTERVAL = 4    # Run the model every 4 frames for smoother detection.

# Initialize the config
config = Config()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using device: {device}")

# --- MODEL DEFINITION (Must be identical to the training script) ---
class VisionTransformerAccidentDetector(nn.Module):
    def __init__(self, num_features, hidden_size=256, dropout=0.5):
        super(VisionTransformerAccidentDetector, self).__init__()
        self.base_model = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=0, global_pool='avg')
        num_video_features = self.base_model.num_features
        for param in self.base_model.parameters():
            param.requires_grad = False
        self.video_gru = nn.GRU(input_size=num_video_features, hidden_size=hidden_size, batch_first=True)
        self.feature_gru = nn.GRU(input_size=num_features, hidden_size=hidden_size // 4, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + hidden_size // 4, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 1)
        )

    def forward(self, video_input, feature_input):
        batch_size, clip_len, C, H, W = video_input.shape
        video_input_reshaped = video_input.view(batch_size * clip_len, C, H, W)
        video_features = self.base_model(video_input_reshaped)
        video_features_seq = video_features.view(batch_size, clip_len, -1)
        _, video_hidden = self.video_gru(video_features_seq)
        _, feature_hidden = self.feature_gru(feature_input)
        video_hidden = video_hidden.squeeze(0)
        feature_hidden = feature_hidden.squeeze(0)
        fused = torch.cat((video_hidden, feature_hidden), dim=1)
        return self.classifier(fused).squeeze(1)

# --- HELPER FUNCTION: Find Involved Vehicles ---
def find_involved_vehicles(buffered_tracks, deceleration_threshold=10.0, iou_threshold=0.05, stop_velocity_threshold=1.0, stop_duration=3):
    involved_ids = set()
    if len(buffered_tracks) < stop_duration: return involved_ids, 0
    df = pd.DataFrame(buffered_tracks); df = df.sort_values(by=['track_id', 'frame_index'])
    df['x_center']=(df['x_min']+df['x_max'])/2; df['y_center']=(df['y_min']+df['y_max'])/2
    df['velocity_x']=df.groupby('track_id')['x_center'].diff().fillna(0).abs()
    df['velocity_y']=df.groupby('track_id')['y_center'].diff().fillna(0).abs()
    df['velocity']=np.sqrt(df['velocity_x']**2 + df['velocity_y']**2)
    df['deceleration'] = -df.groupby('track_id')['velocity'].diff().fillna(0)
    high_decel_ids = set(df[df['deceleration'] > deceleration_threshold]['track_id'])
    stopped_ids = set()
    for track_id, group in df.groupby('track_id'):
        if len(group) >= stop_duration and (group['velocity'].tail(stop_duration) < stop_velocity_threshold).all():
            stopped_ids.add(track_id)
    suspect_ids = high_decel_ids.union(stopped_ids)
    final_frame_df = df[df['frame_index'] == df['frame_index'].max()]
    suspect_df = final_frame_df[final_frame_df['track_id'].isin(suspect_ids)]
    if len(suspect_df) > 1:
        for i, j in combinations(suspect_df.index, 2):
            box1=suspect_df.loc[i,['x_min','y_min','x_max','y_max']].values; box2=suspect_df.loc[j,['x_min','y_min','x_max','y_max']].values
            inter_area = max(0, min(box1[2],box2[2]) - max(box1[0],box2[0])) * max(0, min(box1[3],box2[3]) - max(box1[1],box2[1]))
            if inter_area > 0:
                box1_area=(box1[2]-box1[0])*(box1[3]-box1[1]); box2_area=(box2[2]-box2[0])*(box2[3]-box2[1])
                union_area = box1_area + box2_area - inter_area
                if inter_area / union_area > iou_threshold:
                    involved_ids.add(int(suspect_df.loc[i,'track_id'])); involved_ids.add(int(suspect_df.loc[j,'track_id']))
    involved_ids.update(high_decel_ids)
    return involved_ids, len(involved_ids)

# --- MAIN ANALYSIS WORKFLOW ---
def main():
    print("--- Final Analysis with Rich Features and Bounding Box Labels ---")
    
    event_detector_model = VisionTransformerAccidentDetector(num_features=3).to(device)
    event_detector_model.load_state_dict(torch.load(config.SAVED_MODEL_PATH, map_location=device)); event_detector_model.eval()
    object_tracker_model = YOLO(config.YOLO_MODEL_PATH)

    video_info = sv.VideoInfo.from_video_path(config.INPUT_VIDEO_PATH)
    frame_generator = sv.get_video_frames_generator(source_path=config.INPUT_VIDEO_PATH)
    transform = transforms.Compose([transforms.Resize(config.IMAGE_SIZE), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
    
    frame_buffer, feature_buffer, tracking_buffer = deque(maxlen=config.CLIP_LENGTH), deque(maxlen=config.CLIP_LENGTH), deque(maxlen=config.CLIP_LENGTH + 15)
    accident_detected, event_cooldown = False, 0
    involved_vehicle_ids, num_involved = set(), 0
    
    green_annotator = sv.BoxAnnotator(thickness=2, color=sv.Color.GREEN)
    red_annotator = sv.BoxAnnotator(thickness=2, color=sv.Color.RED)
    label_annotator = sv.LabelAnnotator(text_color=sv.Color.WHITE, text_scale=0.5)

    with sv.VideoSink(config.OUTPUT_VIDEO_PATH, video_info) as sink:
        for frame_index, frame in enumerate(tqdm(frame_generator, total=video_info.total_frames, desc="Analyzing Video")):
            
            track_results = object_tracker_model.track(frame, persist=True, verbose=False)[0]
            detections = sv.Detections.from_ultralytics(track_results)

            # --- Real-Time Rich Feature Calculation ---
            current_frame_tracks = []
            if detections.tracker_id is not None:
                for i in range(len(detections)):
                    tracking_buffer.append({'frame_index': frame_index, 'track_id': detections.tracker_id[i], 'x_min': detections.xyxy[i][0], 'y_min': detections.xyxy[i][1], 'x_max': detections.xyxy[i][2], 'y_max': detections.xyxy[i][3]})
                    current_frame_tracks.append({'track_id': detections.tracker_id[i]})
            
            max_decel, max_iou, max_dir_change = 0, 0, 0
            if len(tracking_buffer) > 1 and len(detections) > 0:
                df = pd.DataFrame(list(tracking_buffer)); df = df.sort_values(by=['track_id', 'frame_index'])
                df['x_center']=(df['x_min']+df['x_max'])/2; df['y_center']=(df['y_min']+df['y_max'])/2
                df[['dx','dy']] = df.groupby('track_id')[['x_center','y_center']].diff().fillna(0)
                df['velocity'] = np.sqrt(df['dx']**2 + df['dy']**2); df['deceleration'] = -df.groupby('track_id')['velocity'].diff().fillna(0)
                v_prev = df.groupby('track_id')[['dx','dy']].shift(1).fillna(0).to_numpy(); v_curr = df[['dx','dy']].to_numpy()
                dot = np.sum(v_prev*v_curr,axis=1); norm_prev=np.linalg.norm(v_prev,axis=1); norm_curr=np.linalg.norm(v_curr,axis=1)
                denom = norm_prev * norm_curr; cos_ang = np.divide(dot, denom, out=np.ones_like(dot), where=denom!=0)
                df['direction_change'] = np.degrees(np.arccos(np.clip(cos_ang, -1.0, 1.0)))
                
                current_frame_df = df[df['frame_index'] == frame_index]
                if not current_frame_df.empty:
                    max_decel = current_frame_df['deceleration'].max(); max_dir_change = current_frame_df['direction_change'].max()
                    if len(current_frame_df) > 1:
                        for i, j in combinations(current_frame_df.index, 2):
                            box1=current_frame_df.loc[i,['x_min','y_min','x_max','y_max']].values; box2=current_frame_df.loc[j,['x_min','y_min','x_max','y_max']].values
                            inter_area = max(0, min(box1[2],box2[2])-max(box1[0],box2[0])) * max(0, min(box1[3],box2[3])-max(box1[1],box2[1]))
                            if inter_area > 0:
                                b1_area=(box1[2]-box1[0])*(box1[3]-box1[1]); b2_area=(box2[2]-box2[0])*(box2[3]-box2[1])
                                union = b1_area + b2_area - inter_area
                                max_iou = max(max_iou, inter_area / union if union > 0 else 0)
                max_decel, max_iou, max_dir_change = np.nan_to_num(max_decel), np.nan_to_num(max_iou), np.nan_to_num(max_dir_change)
            feature_buffer.append([max_decel, max_iou, max_dir_change])
            
            pil_frame = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            frame_buffer.append(transform(pil_frame))

            if len(frame_buffer) == config.CLIP_LENGTH and frame_index % config.PREDICTION_INTERVAL == 0:
                video_tensor = torch.stack(list(frame_buffer)).unsqueeze(0).to(device)
                feature_tensor = torch.tensor(list(feature_buffer), dtype=torch.float32).unsqueeze(0).to(device)
                with torch.no_grad():
                    output = event_detector_model(video_tensor, feature_tensor)
                    prediction_score = torch.sigmoid(output).item()
                print(f"Frame {frame_index}, Score: {prediction_score:.4f}")
                if prediction_score >= config.DETECTION_THRESHOLD and not accident_detected:
                    accident_detected = True; event_cooldown = video_info.fps * 5
                    involved_vehicle_ids, num_involved = find_involved_vehicles(list(tracking_buffer))
            
            annotated_frame = frame.copy()
            if detections.tracker_id is not None:
                involved_mask = np.array([tid in involved_vehicle_ids for tid in detections.tracker_id])
                labels = [f"ID {tracker_id}" for tracker_id in detections.tracker_id]
                annotated_frame = green_annotator.annotate(scene=annotated_frame, detections=detections[~involved_mask])
                annotated_frame = red_annotator.annotate(scene=annotated_frame, detections=detections[involved_mask])
                annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)

            if accident_detected:
                sv.draw_text(annotated_frame, "ACCIDENT DETECTED", sv.Point(50, 50), text_scale=2, text_thickness=2, text_color=sv.Color.RED, background_color=sv.Color.WHITE)
                sv.draw_text(annotated_frame, f"Involved Vehicles: {num_involved}", sv.Point(50, 100), text_scale=1.5, text_thickness=2, text_color=sv.Color.BLACK, background_color=sv.Color.WHITE)
            
            if event_cooldown > 0: event_cooldown -= 1
            else:
                if accident_detected: accident_detected, involved_vehicle_ids, num_involved = False, set(), 0
            sink.write_frame(annotated_frame)

    print(f"\n--- Analysis Complete ---"); print(f"✓ Final annotated video saved to: {config.OUTPUT_VIDEO_PATH}")

if __name__ == "__main__":
    main()

✓ Using device: cuda
--- Final Analysis with Rich Features and Bounding Box Labels ---
PRO TIP 💡 Replace 'model=yolov5x.pt' with new 'model=yolov5xu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.



Analyzing Video:  21%|██        | 33/157 [00:11<00:45,  2.74it/s]

Frame 32, Score: 0.6898


Analyzing Video:  24%|██▎       | 37/157 [00:13<00:46,  2.60it/s]

Frame 36, Score: 0.6858


...

Frame 148, Score: 0.7902


Analyzing Video:  97%|█████████▋| 153/157 [01:00<00:01,  2.39it/s]

Frame 152, Score: 0.7910


Analyzing Video: 100%|██████████| 157/157 [01:02<00:00,  2.52it/s]

Frame 156, Score: 0.7876

--- Analysis Complete ---
✓ Final annotated video saved to: /kaggle/working/final_analysis_output.mp4
